In [1]:
# Install necessary packages
!pip install tensorflow keras numpy pandas scikit-learn gensim requests tqdm nltk

# Download and install ISRI Stemmer (from NLTK)
import nltk
nltk.download('arabic_reshaper')
nltk.download('stopwords')
from nltk.stem.isri import ISRIStemmer


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 576.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.3/106.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing ins

[nltk_data] Error loading arabic_reshaper: Package 'arabic_reshaper'
[nltk_data]     not found in index
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import gensim
import requests
import zipfile
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from nltk.stem.isri import ISRIStemmer
from nltk.corpus import stopwords


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import pandas as pd

# Define the path to your CSV file in Google Drive
csv_path = "/content/drive/My Drive/ANLP/SEC05/Arabic Sentiment Analysis Dataset - SS2030.csv"  # Update with the actual path

# Read the CSV file
df = pd.read_csv(csv_path, encoding="utf-8")  # Use 'utf-8-sig' if needed
df = df.dropna()  # Remove missing values

# Display sample
print(df.head())


                                                text  Sentiment
0            حقوق المرأة 💚💚💚 https://t.co/Mzf90Ta5g1          1
1  RT @___IHAVENOIDEA: حقوق المرأة في الإسلام. ht...          1
2  RT @saud_talep: Retweeted لجنة التنمية بشبرا (...          1
3  RT @MojKsa: حقوق المرأة التي تضمنها لها وزارة ...          1
4  RT @abm112211: ولي امر الزوجة او ولي الزوجة او...          1


In [5]:
# Initialize ISRI Stemmer
stemmer = ISRIStemmer()
stop_words = set(stopwords.words("arabic"))

# Function to preprocess Arabic text
def preprocess_arabic_text(text):
    text = str(text)  # Ensure it's a string
    text = text.replace("،", "").replace(".", "").replace("؟", "").replace("!", "")
    text = text.replace("\n", " ").replace("\r", " ")
    words = text.split()  # Tokenize manually (since ISRI doesn't tokenize)
    words = [stemmer.stem(word) for word in words if word not in stop_words]  # Stem and remove stopwords
    return " ".join(words)

# Apply preprocessing
df["processed_text"] = df["text"].apply(preprocess_arabic_text)
df.columns = df.columns.str.strip()
df = df.rename(columns={"Sentiment": "label"})

# Display processed data
df.head()


,text,label,processed_text
0,حقوق المرأة 💚💚💚 https://t.co/Mzf90Ta5g1,1,حقق رأة 💚💚💚 https://tco/Mzf90Ta5g1
1,RT @___IHAVENOIDEA: حقوق المرأة في الإسلام. ht...,1,RT @___IHAVENOIDEA: حقق رأة سلم https://tco/ps...
2,RT @saud_talep: Retweeted لجنة التنمية بشبرا (...,1,RT @saud_talep: Retweeted لجن نمي شبر (@Shubra...
3,RT @MojKsa: حقوق المرأة التي تضمنها لها وزارة ...,1,RT @MojKsa: حقق رأة تضم وزر عدل https://tco/QU...
4,RT @abm112211: ولي امر الزوجة او ولي الزوجة او...,1,RT @abm112211: ولي امر زوج او ولي زوج او ولي ر...


In [6]:
# Tokenization
tokenizer = Tokenizer(num_words=10000)  # Keep top 10K words
tokenizer.fit_on_texts(df["processed_text"])

# Convert text to sequences
X_sequences = tokenizer.texts_to_sequences(df["processed_text"])
X_padded = pad_sequences(X_sequences, maxlen=100)  # Standardize length

# Labels
y = np.array(df["label"])

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_padded, y, test_size=0.2, random_state=42)


In [7]:
# Define Trainable Embedding Model
trainable_model = Sequential([
    Embedding(input_dim=10000, output_dim=300, input_length=100, trainable=True),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

# Compile the model
trainable_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
trainable_model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

# Evaluate
trainable_loss, trainable_acc = trainable_model.evaluate(X_test, y_test)
print(f"Trainable Embedding Model Accuracy: {trainable_acc:.2f}")


Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


107/107 ━━━━━━━━━━━━━━━━━━━━ 23s 179ms/step - accuracy: 0.7188 - loss: 0.5467 - val_accuracy: 0.8566 - val_loss: 0.3537
Epoch 2/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 17s 156ms/step - accuracy: 0.9509 - loss: 0.1273 - val_accuracy: 0.8731 - val_loss: 0.3269
Epoch 3/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 17s 160ms/step - accuracy: 0.9906 - loss: 0.0422 - val_accuracy: 0.8519 - val_loss: 0.5036
Epoch 4/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 17s 157ms/step - accuracy: 0.9967 - loss: 0.0139 - val_accuracy: 0.8566 - val_loss: 0.6059
Epoch 5/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 17s 158ms/step - accuracy: 0.9962 - loss: 0.0159 - val_accuracy: 0.8496 - val_loss: 0.5953
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.8257 - loss: 0.6966
Trainable Embedding Model Accuracy: 0.85


In [8]:
# Download AraVec Word2Vec Model
aravec_url = "https://bakrianoo.ewr1.vultrobjects.com/aravec/full_grams_cbow_300_twitter.zip"
aravec_zip_path = "aravec.zip"
aravec_model_path = "full_grams_cbow_300_twitter.mdl"

if not os.path.exists(aravec_model_path):
    response = requests.get(aravec_url, stream=True)
    with open(aravec_zip_path, "wb") as file:
        for chunk in tqdm(response.iter_content(chunk_size=1024)):
            file.write(chunk)

    # Extract the zip file
    with zipfile.ZipFile(aravec_zip_path, "r") as zip_ref:
        zip_ref.extractall("aravec")

# Load Word2Vec model
word2vec_model = gensim.models.Word2Vec.load('./aravec/full_grams_cbow_300_twitter.mdl')


3247588it [00:35, 91383.82it/s] 


In [9]:
word2vec_model = gensim.models.Word2Vec.load('./aravec/full_grams_cbow_300_twitter.mdl')

In [10]:
embedding_dim = 300
embedding_matrix = np.zeros((len(tokenizer.word_index) + 1, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in word2vec_model.wv:  # Corrected: Use .wv to check vocabulary
        embedding_matrix[i] = word2vec_model.wv[word]  # Corrected: Use .wv to access word vectors

print(f"Embedding matrix shape: {embedding_matrix.shape}")


Embedding matrix shape: (11710, 300)


In [11]:
# Define LSTM Model with Pretrained Embeddings
pretrained_model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=embedding_dim,
              weights=[embedding_matrix], input_length=100, trainable=False),  # Use pretrained embeddings (Frozen)
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile model
pretrained_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Display model summary
pretrained_model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │       3,513,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_2 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,513,000 (13.40 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 3,513,000 (13.40 MB)

In [12]:
# Train the model
pretrained_model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

# Evaluate model
pretrained_loss, pretrained_acc = pretrained_model.evaluate(X_test, y_test)
print(f"Pretrained Word2Vec Model Accuracy: {pretrained_acc:.2f}")


Epoch 1/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 22s 174ms/step - accuracy: 0.7081 - loss: 0.5336 - val_accuracy: 0.8120 - val_loss: 0.3891
Epoch 2/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 18s 171ms/step - accuracy: 0.8829 - loss: 0.2669 - val_accuracy: 0.8343 - val_loss: 0.3698
Epoch 3/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 18s 171ms/step - accuracy: 0.9314 - loss: 0.1794 - val_accuracy: 0.8390 - val_loss: 0.3783
Epoch 4/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 18s 172ms/step - accuracy: 0.9508 - loss: 0.1287 - val_accuracy: 0.8496 - val_loss: 0.3836
Epoch 5/5
107/107 ━━━━━━━━━━━━━━━━━━━━ 18s 170ms/step - accuracy: 0.9760 - loss: 0.0757 - val_accuracy: 0.8555 - val_loss: 0.4293
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.8577 - loss: 0.4420
Pretrained Word2Vec Model Accuracy: 0.86


In [13]:
print(f"Trainable Embedding Model Accuracy: {trainable_acc:.2f}")
print(f"Pretrained Word2Vec Model Accuracy: {pretrained_acc:.2f}")


Trainable Embedding Model Accuracy: 0.85
Pretrained Word2Vec Model Accuracy: 0.86
